# Sixth Week Assignment — Building a WaveNet


## 1. Katman sınıfları ve Sequential


In [ ]:
import json
from pathlib import Path

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from makemore_wavenet import *

ROOT = Path(".")
results = json.loads((ROOT / "results.json").read_text(encoding="utf-8"))
rows = {row["model"]: row for row in results["results"]}

In [ ]:
from makemore_wavenet import Linear, BatchNorm1d, Tanh, Embedding, Flatten, Sequential

words = load_words(ROOT / "names.txt")
data3 = prepare_data(words, block_size=3)
model = build_mlp(len(data3["stoi"]), block_size=3, n_embd=10, n_hidden=200)
print(model.layers)
print("parameters:", count_parameters(model))
model.train()
print("eğitim kipi:", [layer.training for layer in model.layers if isinstance(layer, BatchNorm1d)])
model.eval()
print("tahmin kipi:", [layer.training for layer in model.layers if isinstance(layer, BatchNorm1d)])

# Eğitim döngüsü yalnızca Sequential arayüzünü bilir.
plt.plot(results["loss_curves"]["context 3 MLP"])
plt.xlabel("1000 adımlık grup")
plt.ylabel("ortalama log10(loss)")
plt.title("Düzeltilmiş loss eğrisi")

Loss eğrisi `log10` loss değerlerinin 1000 adımlık ortalamasıyla çizilir.


## 2. Bağlam 3 ve bağlam 8


In [ ]:
base = rows["context 3 MLP"]
flat8 = rows["context 8 flat MLP"]
parameter_gain = flat8["parameters"] - base["parameters"]
loss_gain = base["dev_loss"] - flat8["dev_loss"]
print(f"Bağlam 3: {base['parameters']} parametre, dev loss {base['dev_loss']:.4f}")
print(f"Bağlam 8: {flat8['parameters']} parametre, dev loss {flat8['dev_loss']:.4f}")
print(f"Parametre artışı: {parameter_gain} (%{100 * parameter_gain / base['parameters']:.1f})")
print(f"Dev loss düşüşü: {loss_gain:.4f}")

Bağlam büyüdü; düz model hâlâ sekiz harfi tek adımda karıştırıyor.


## 3. Hiyerarşik WaveNet ve şekiller


In [ ]:
for layer, shape in results["shapes"]:
    print(f"{layer:20s} {tuple(shape)}")

Her `FlattenConsecutive(2)` zaman eksenini yarıya indirip komşu iki konumun kanallarını birleştirir. Üç katmanda `8 → 4 → 2 → 1` olur.


## 4. Üç boyutlu BatchNorm


In [ ]:
x = torch.randn(32, 4, 68)
wrong_mean = x.mean(0, keepdim=True)
correct_mean = x.mean((0, 1), keepdim=True)
print("girdi:", tuple(x.shape))
print("yanlış ortalama (yalnız batch):", tuple(wrong_mean.shape))
print("doğru ortalama (batch + time):", tuple(correct_mean.shape))

wrong = rows["context 8 WaveNet wrong BN"]
fixed = rows["context 8 WaveNet"]
print(f"yanlış BN dev loss: {wrong['dev_loss']:.4f}")
print(f"doğru BN dev loss:  {fixed['dev_loss']:.4f}")

`(B, T, C)` için yalnız `B` değil, `(B, T)` düzlemi normalize edilmeli; doğru indirgeme eksenleri `(0, 1)`'dir.


## 5. Sonuç tablosu


In [ ]:
selected = [
    "context 3 MLP",
    "context 8 flat MLP",
    "context 8 WaveNet scaled",
]
print(f"{'model':32s} {'parameters':>12s} {'dev loss':>10s}")
for name in selected:
    row = rows[name]
    print(f"{name:32s} {row['parameters']:12d} {row['dev_loss']:10.4f}")

## 6. Türkçe isimler


In [ ]:
mlp = rows["Turkish context 3 MLP"]
wavenet = rows["Turkish context 8 WaveNet"]
gain = mlp["dev_loss"] - wavenet["dev_loss"]
print(f"Hafta 4 tarzı bağlam-3 MLP: {mlp['dev_loss']:.4f}")
print(f"Bağlam-8 WaveNet:           {wavenet['dev_loss']:.4f}")
print(f"Kazanç: {gain:.4f} (%{100 * gain / mlp['dev_loss']:.1f})")
print("\nÜretilen örnekler:")
print(", ".join(results["turkish_samples"]))

Türkçe dev loss `2.1104 → 1.9983`; sekiz karakterlik hiyerarşik bağlam uzun hece ve ek örüntülerini daha iyi taşıdı.


## 7. Ek: deney düzeneği


In [ ]:
configs = [
    {"name": "context 3 MLP", "block_size": 3, "kind": "mlp", "n_embd": 10, "n_hidden": 200},
    {"name": "context 8 flat MLP", "block_size": 8, "kind": "mlp", "n_embd": 10, "n_hidden": 200},
    {"name": "context 8 WaveNet", "block_size": 8, "kind": "wavenet", "n_embd": 10, "n_hidden": 68},
    {"name": "context 8 WaveNet scaled", "block_size": 8, "kind": "wavenet", "n_embd": 24, "n_hidden": 128},
]

for config in configs:
    print(config)

print("Tam deney: python run_experiments.py --steps 30000")